libaraies

In [81]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, resample_poly
import neurokit2 as nk
from scipy.signal import find_peaks
import seaborn as sns


# Pandas display settings (optional)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

load files

In [82]:
# RESP FILE PATHS
RI1_3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5"
RI2_3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5"
BLRI_s3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5"
BLRI_s4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5"
RI_4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5"
RI2_4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5"
RI1_2_3_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5"
RI2_2_3_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5"
RI1_4_8_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5"
RI2_4_8_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5"

# BORIS CSVs (no comma at the end or it makes a tuple)
RI1_3_6_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv")
RI2_3_6_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv")
RI1_4_7_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv")
RI2_4_7_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv")
RI1_2_3_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_3_p5_3_nRB3_20250622_104059.1.csv")
RI2_2_3_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_3_p5_3_nRB3_20250622_1102116.1.csv")
RI1_4_8_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_8_p5_1_nRB3_HEEPS.csv")
RI2_4_8_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv")

In [83]:
# RI2 H5 paths
ri2_h5_paths = {
    "RI2_3_6": RI2_3_6_h5_path,
    "RI2_4_7": RI2_4_7_h5_path,
    "RI2_2_3": RI2_2_3_h5_path,
    "RI2_4_8": RI2_4_8_h5_path,
}

# RI2 BORIS data
ri2_boris_data = {
    "RI2_3_6": RI2_3_6_boris.copy(),
    "RI2_4_7": RI2_4_7_boris.copy(),
    "RI2_2_3": RI2_2_3_boris.copy(),
    "RI2_4_8": RI2_4_8_boris.copy(),
}


load_clean_resp_signal function for loading and preprocessing respiration data:

Reads raw resp signal and metadata from .h5

Computes original sampling rate (fs)

Applies a low-pass filter before downsampling to avoid aliasing

Downsamples the signal to your target_rate (default 100 Hz)

Applies a bandpass filter (0.1–20 Hz) with NeuroKit to clean the signal further

Creates a time vector that matches the cleaned signal length and sampling rate

This function returns:
rsp_cleaned: the cleaned, filtered, downsampled respiratory signal array

time_vector: time points corresponding to each sample in seconds

target_rate: the sampling rate after downsampling (e.g., 100 Hz)

You can then pass these outputs to functions like:
get_percent_change_resp_rate(signal=rsp_cleaned, times=time_vector, sniff_start, sniff_end)

get_delta_resp_rate(signal=rsp_cleaned, times=time_vector, sniff_end)

In [84]:
def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Loads, filters, downsamples respiration from .h5, returns cleaned signal and time vector.
    """
    try:
        with h5py.File(h5_file, 'r') as f:
            resp = f['resp'][:].flatten()
            ekg_meta = dict(f['ekg_metadata'].attrs)
        duration_sec = ekg_meta['duration_sec']
        fs = len(resp) / duration_sec
    except Exception as e:
        print(f"Error loading {h5_file}: {e}")
        return None, None, None

    # Pre-filter before downsampling
    nyquist = fs / 2
    norm_cutoff = (target_rate / 2) / nyquist
    b, a = butter(N=4, Wn=norm_cutoff, btype='low')
    filtered_resp = filtfilt(b, a, resp)

    # Downsample
    downsample_factor = int(fs // target_rate)
    downsampled = resample_poly(filtered_resp, up=1, down=downsample_factor)

    # Bandpass filter with neurokit
    rsp_cleaned = nk.signal_filter(
        downsampled,
        lowcut=0.1,
        highcut=20,
        method="butterworth",
        sampling_rate=target_rate,
        order=2
    )

    # Generate matching time vector
    time_vector = np.arange(len(rsp_cleaned)) / target_rate

    return rsp_cleaned, time_vector, target_rate

In [85]:
# Dictionary of H5 files with labels for tracking
h5_paths = {
    "RI1_3_6": RI1_3_6_h5_path,
    "RI2_3_6": RI2_3_6_h5_path,
    "BLRI_s3_6": BLRI_s3_6_h5_path,
    "BLRI_s4_7": BLRI_s4_7_h5_path,
    "RI1_4_7": RI_4_7_h5_path,
    "RI2_4_7": RI2_4_7_h5_path,
    "RI1_2_3": RI1_2_3_h5_path,
    "RI2_2_3": RI2_2_3_h5_path,
    "RI1_4_8": RI1_4_8_h5_path,
    "RI2_4_8": RI2_4_8_h5_path
}

# Create a dictionary to store results
resp_data = {}

# Loop through files and apply the function
for label, path in h5_paths.items():
    print(f"Processing {label}...")
    signal, time, rate = load_clean_resp_signal(path)
    if signal is not None:
        resp_data[label] = {
            "signal": signal,
            "time": time,
            "sampling_rate": rate
        }
    else:
        print(f"Skipping {label}: failed to load.")

print("All available signals loaded.")


Processing RI1_3_6...
Processing RI2_3_6...
Processing BLRI_s3_6...
Processing BLRI_s4_7...
Processing RI1_4_7...
Processing RI2_4_7...
Processing RI1_2_3...
Processing RI2_2_3...
Processing RI1_4_8...
Processing RI2_4_8...
All available signals loaded.


In [86]:
for subject, df in ri2_boris_data.items():
    count = df[(df['Behavior'] == 'facial sniffing') & (df['Subject'] == 'social_agent')].shape[0]
    print(f"{subject} facial sniffing count: {count}")


RI2_3_6 facial sniffing count: 26
RI2_4_7 facial sniffing count: 21
RI2_2_3 facial sniffing count: 12
RI2_4_8 facial sniffing count: 38


In [87]:
for subject, df in ri2_boris_data.items():
    count = df[df['Behavior'] == 'fighting'].shape[0]
    print(f"{subject} fight count: {count}")


RI2_3_6 fight count: 18
RI2_4_7 fight count: 30
RI2_2_3 fight count: 14
RI2_4_8 fight count: 28


In [88]:
from scipy.signal import find_peaks

for subject, resp_file in ri2_h5_paths.items():
    resp_signal, resp_times, fs = load_clean_resp_signal(resp_file, target_rate=100)
    if resp_signal is None:
        print(f"{subject}: Resp signal failed to load.")
        continue
    peaks, _ = find_peaks(resp_signal, distance=int(fs * 0.15))
    print(f"{subject}: {len(peaks)} peaks found in respiration signal.")


RI2_3_6: 2788 peaks found in respiration signal.
RI2_4_7: 2819 peaks found in respiration signal.
RI2_2_3: 2718 peaks found in respiration signal.
RI2_4_8: 2846 peaks found in respiration signal.


In [89]:
for label, path in h5_paths.items():
    print(f"{label} is type {type(path)}")


RI1_3_6 is type <class 'str'>
RI2_3_6 is type <class 'str'>
BLRI_s3_6 is type <class 'str'>
BLRI_s4_7 is type <class 'str'>
RI1_4_7 is type <class 'str'>
RI2_4_7 is type <class 'str'>
RI1_2_3 is type <class 'str'>
RI2_2_3 is type <class 'str'>
RI1_4_8 is type <class 'str'>
RI2_4_8 is type <class 'str'>


In [90]:
for key in resp_data:
    print(f"{key}: {len(resp_data[key]['signal'])} samples at {resp_data[key]['sampling_rate']} Hz")


RI1_3_6: 60328 samples at 100 Hz
RI2_3_6: 61001 samples at 100 Hz
BLRI_s3_6: 60629 samples at 100 Hz
BLRI_s4_7: 60409 samples at 100 Hz
RI1_4_7: 62916 samples at 100 Hz
RI2_4_7: 60647 samples at 100 Hz
RI1_2_3: 60295 samples at 100 Hz
RI2_2_3: 60921 samples at 100 Hz
RI1_4_8: 60484 samples at 100 Hz
RI2_4_8: 60324 samples at 100 Hz


get_sniff_respiratory_rate function is good for calculating the breathing rate during a sniff bout:

It creates a mask to select the respiratory signal samples between sniff_start and sniff_end.

It uses find_peaks to detect inhalation peaks in that signal segment, with a minimum peak distance of 0.2 seconds (to avoid counting the same breath twice).

It calculates the duration of the sniff bout.

It returns the breathing rate as the number of detected peaks divided by the duration (giving rate in Hz).

A few things to watch out for:
Sampling rate consistency: Make sure the times array and the signal array are aligned and that sampling_rate matches the sampling frequency of the signal.

Peak detection threshold: You might want to tweak the distance or add a height parameter in find_peaks if you get too many or too few peaks.

Edge cases: If duration is zero or very small, it returns np.nan, which is good.



In [91]:
def get_sniff_respiratory_rate(signal, time, sniff_start, sniff_end, sampling_rate=100):
    """
    Compute breathing rate (Hz) during a sniff bout.
    """
    sniff_mask = (time >= sniff_start) & (time < sniff_end)
    signal_sniff = signal[sniff_mask]

    peaks, _ = find_peaks(signal_sniff, distance=sampling_rate * 0.2)

    duration = sniff_end - sniff_start
    rate = len(peaks) / duration if duration > 0 else np.nan

    return rate

In [92]:
def extract_agent_face_sniffs(df):
    """
    Filters BORIS dataframe for facial sniffing initiated by the social agent.
    """
    df = df.copy()
    df['Start (s)'] = df['Start (s)'].astype(float)
    df['Stop (s)'] = df['Stop (s)'].astype(float)
    df = df.sort_values(by='Start (s)').reset_index(drop=True)

    # Updated filters
    agent_mask = df['Subject'].str.lower() == 'social_agent'
    sniff_mask = df['Behavior'].str.lower() == 'facial sniffing'

    return df[agent_mask & sniff_mask]


In [93]:
agent_sniffs = extract_agent_face_sniffs(RI2_3_6_boris)
print(agent_sniffs[['Subject', 'Behavior', 'Start (s)', 'Stop (s)']])


          Subject         Behavior  Start (s)  Stop (s)
12   social_agent  facial sniffing     25.600    26.100
23   social_agent  facial sniffing     68.767    69.600
36   social_agent  facial sniffing     91.900    92.267
40   social_agent  facial sniffing    153.900   154.400
41   social_agent  facial sniffing    156.233   156.567
43   social_agent  facial sniffing    195.433   195.767
46   social_agent  facial sniffing    196.167   197.500
53   social_agent  facial sniffing    218.300   219.367
59   social_agent  facial sniffing    226.433   226.900
64   social_agent  facial sniffing    234.867   235.367
67   social_agent  facial sniffing    249.733   250.567
68   social_agent  facial sniffing    257.833   258.367
70   social_agent  facial sniffing    267.400   268.933
71   social_agent  facial sniffing    269.767   270.600
75   social_agent  facial sniffing    273.367   274.200
78   social_agent  facial sniffing    282.033   283.067
79   social_agent  facial sniffing    288.200   

In [94]:
def calculate_percent_change_ibi(signal, time, sniff_start, sampling_rate=100):
    """
    Calculates % change in IBI using:
    - Baseline: -1.5s to -0.5s before sniff
    - Post-sniff: +0.5s to +1.5s after sniff

    Returns:
        float: % change in IBI, or None if not enough peaks
    """
    # Define time windows
    baseline_mask = (time >= sniff_start - 1.5) & (time < sniff_start - 0.5)
    post_mask = (time >= sniff_start + 0.5) & (time < sniff_start + 1.5)

    baseline_segment = signal[baseline_mask]
    post_segment = signal[post_mask]

    min_distance = int(0.2 * sampling_rate)  # at least 200 ms between peaks

    baseline_peaks, _ = find_peaks(baseline_segment, distance=min_distance)
    post_peaks, _ = find_peaks(post_segment, distance=min_distance)

    baseline_peak_times = time[baseline_mask][baseline_peaks]
    post_peak_times = time[post_mask][post_peaks]

    # Calculate IBIs
    baseline_ibis = np.diff(baseline_peak_times)
    post_ibis = np.diff(post_peak_times)

    if len(baseline_ibis) == 0 or len(post_ibis) == 0:
        return None

    baseline_mean_ibi = np.mean(baseline_ibis)
    post_mean_ibi = np.mean(post_ibis)

    percent_change = ((post_mean_ibi - baseline_mean_ibi) / baseline_mean_ibi) * 100
    return percent_change


In [95]:
sniff_start = agent_sniffs.iloc[0]["Start (s)"]
signal = resp_data["RI2_3_6"]["signal"]
time = resp_data["RI2_3_6"]["time"]
sr = resp_data["RI2_3_6"]["sampling_rate"]

calculate_percent_change_ibi(signal, time, sniff_start, sr)


13.409961685823385

baseline_mask = (time >= sniff_start - 1.5) & (time < sniff_start - 0.5)
post_mask     = (time >= sniff_start + 0.5) & (time < sniff_start + 1.5)

🟢 These define time windows:

Baseline window = from 1.5s before to 0.5s before the sniff

Post-sniff window = from 0.5s after to 1.5s after the sniff

So we ignore the sniff itself (−0.5 to +0.5), and only look at clean periods before and after.

baseline_peaks, _ = find_peaks(signal[baseline_mask], distance=0.2 * sampling_rate)
post_peaks, _     = find_peaks(signal[post_mask], distance=0.2 * sampling_rate)

🟢 This finds breath peaks in the respiration signal during each window:

find_peaks(...) detects upward spikes (inhalations or cycle peaks)

distance=0.2 * sampling_rate means it skips peaks closer than 200 ms apart ask Dan and Nancy about this threshold. 

baseline_times = time[baseline_mask][baseline_peaks]
post_times     = time[post_mask][post_peaks]

🟢 This gives you the actual time values of those breath peaks (in seconds).

So you now have the times of each breath in each window.

baseline_ibis = np.diff(baseline_times)
post_ibis     = np.diff(post_times)

🟢 This calculates the inter-breath intervals (IBIs):

np.diff(...) gives the time between consecutive breaths

if len(baseline_ibis) == 0 or len(post_ibis) == 0:
    return None

🛑 Safety check — if no IBIs found in either window (e.g. too few breaths), skip it.

baseline_rate = 1 / np.mean(baseline_ibis)
post_rate     = 1 / np.mean(post_ibis)

✅ This converts IBI to breathing rate (Hz):

Rate = 1 / mean IBI

Units = breaths per second

baseline_rate = 1 / np.mean(baseline_ibis)
post_rate     = 1 / np.mean(post_ibis)

✅ This converts IBI to breathing rate (Hz):

Rate = 1 / mean IBI

Units = breaths per second

rate_change = (post_rate - baseline_rate) / baseline_rate * 100

🧮 This computes % change in breathing rate:

Positive = breathing sped up

Negative = breathing slowed down

🔁 Summary of Logic:

Get time windows before and after sniff

Find breaths in each window

Calculate IBI → convert to rate

Compare post vs. baseline rate

Return % change

In [96]:
def percent_change_resp_rate(signal, time, sniff_start, sampling_rate=100):
    baseline_mask = (time >= sniff_start - 1.5) & (time < sniff_start - 0.5)
    post_mask = (time >= sniff_start + 0.5) & (time < sniff_start + 1.5)

    baseline_peaks, _ = find_peaks(signal[baseline_mask], distance=0.2 * sampling_rate)
    post_peaks, _ = find_peaks(signal[post_mask], distance=0.2 * sampling_rate)

    baseline_times = time[baseline_mask][baseline_peaks]
    post_times = time[post_mask][post_peaks]

    baseline_ibis = np.diff(baseline_times)
    post_ibis = np.diff(post_times)

    if len(baseline_ibis) == 0 or len(post_ibis) == 0:
        return None

    baseline_rate = 1 / np.mean(baseline_ibis)
    post_rate = 1 / np.mean(post_ibis)

    rate_change = (post_rate - baseline_rate) / baseline_rate * 100
    return rate_change


In [97]:
sniff_start = agent_sniffs.iloc[0]["Start (s)"]
signal = resp_data["RI2_3_6"]["signal"]
time = resp_data["RI2_3_6"]["time"]
sr = resp_data["RI2_3_6"]["sampling_rate"]

ibi_pct = calculate_percent_change_ibi(signal, time, sniff_start, sr)
rate_pct = percent_change_resp_rate(signal, time, sniff_start, sr)

print(f"% Change in IBI:   {ibi_pct:.2f}%")
print(f"% Change in Rate: {-ibi_pct:.2f}%  <-- inverted")
print(f"% Change in Rate (from rate): {rate_pct:.2f}%")


% Change in IBI:   13.41%
% Change in Rate: -13.41%  <-- inverted
% Change in Rate (from rate): -11.82%


In [98]:
def extract_agent_face_sniffs(df):
    """
    Filters BORIS dataframe for 'Face Sniff' behaviors initiated by the social agent.
    Assumes 'Behavior' and 'Subject' columns exist.
    """
    df = df.copy()
    df['Start (s)'] = df['Start (s)'].astype(float)
    df['Stop (s)'] = df['Stop (s)'].astype(float)
    df = df.sort_values(by='Start (s)').reset_index(drop=True)
    
    # Adjust these string filters to match your BORIS labels
    mask = (df['Behavior'].str.contains("Face Sniff", case=False)) & \
           (df['Subject'].str.lower().str.contains("agent"))
    return df[mask]


In [99]:
def extract_agent_sniff_events(df):
    """
    Filter BORIS DataFrame for agent-initiated sniffing events (face, body, anogenital).
    """
    sniff_behaviors = ["facial sniffing", "body sniffing", "anogenital sniffing"]
    
    # Ensure consistent capitalization (BORIS might use different cases)
    df["Behavior"] = df["Behavior"].str.lower()
    df["Subject"] = df["Subject"].str.lower()

    agent_sniffs = df[
        (df["Subject"] == "social_agent") &
        (df["Behavior"].isin(sniff_behaviors))
    ].copy()

    return agent_sniffs.reset_index(drop=True)


In [101]:
agent_events = extract_agent_sniff_events(RI2_3_6_boris)
print(agent_events[["Behavior", "Start (s)"]])



               Behavior  Start (s)
0       facial sniffing     25.600
1         body sniffing     26.167
2   anogenital sniffing     26.600
3       facial sniffing     68.767
4       facial sniffing     91.900
5       facial sniffing    153.900
6       facial sniffing    156.233
7       facial sniffing    195.433
8         body sniffing    195.767
9       facial sniffing    196.167
10        body sniffing    198.333
11      facial sniffing    218.300
12        body sniffing    219.433
13      facial sniffing    226.433
14        body sniffing    228.433
15  anogenital sniffing    234.167
16      facial sniffing    234.867
17  anogenital sniffing    235.567
18      facial sniffing    249.733
19      facial sniffing    257.833
20      facial sniffing    267.400
21      facial sniffing    269.767
22      facial sniffing    273.367
23      facial sniffing    282.033
24      facial sniffing    288.200
25      facial sniffing    290.133
26      facial sniffing    318.700
27      facial sniff

In [102]:
# Store results in a list
results = []

# Get respiration info for this trial
resp_entry = resp_data["RI2_3_6"]
signal = resp_entry["signal"]
time = resp_entry["time"]
sr = resp_entry["sampling_rate"]

# Loop through all agent sniff events
for idx, row in agent_events.iterrows():
    behavior = row["Behavior"]
    sniff_start = row["Start (s)"]

    # Calculate % change in rate using your function
    rate_change = percent_change_resp_rate(signal, time, sniff_start, sr)

    # Skip if calculation failed (e.g., not enough breaths)
    if rate_change is None:
        continue

    # Store result
    results.append({
        "Trial": "RI2_3_6",
        "Behavior": behavior,
        "Sniff Time (s)": sniff_start,
        "% Change Rate": rate_change
    })


In [103]:
rate_df = pd.DataFrame(results)
print(rate_df)


      Trial             Behavior  Sniff Time (s)  % Change Rate
0   RI2_3_6      facial sniffing          25.600  -1.182432e+01
1   RI2_3_6        body sniffing          26.167  -7.602339e+00
2   RI2_3_6  anogenital sniffing          26.600   6.578947e+00
3   RI2_3_6      facial sniffing          68.767  -4.000000e+00
4   RI2_3_6      facial sniffing          91.900   2.740741e+01
5   RI2_3_6      facial sniffing         153.900   3.000000e+01
6   RI2_3_6      facial sniffing         156.233  -2.840909e+01
7   RI2_3_6      facial sniffing         195.433  -2.266667e+01
8   RI2_3_6        body sniffing         195.767  -1.919192e+01
9   RI2_3_6      facial sniffing         196.167   0.000000e+00
10  RI2_3_6        body sniffing         198.333  -1.534392e+01
11  RI2_3_6      facial sniffing         218.300   1.612903e+00
12  RI2_3_6        body sniffing         219.433  -2.394366e+01
13  RI2_3_6      facial sniffing         226.433  -3.493976e+01
14  RI2_3_6        body sniffing        

In [ ]:
RI2_3_6_boris.columns

Index(['Observation id', 'Observation date', 'Description', 'Observation type',
       'Source', 'Time offset (s)', 'Coding duration', 'Media duration (s)',
       'FPS (frame/s)', 'Subject',
       'Observation duration by subject by observation', 'Behavior',
       'Behavioral category', 'Behavior type', 'Start (s)', 'Stop (s)',
       'Duration (s)', 'Media file name', 'Image index start',
       'Image index stop', 'Image file path start', 'Image file path stop',
       'Comment start', 'Comment stop'],
      dtype='object')

In [ ]:
agent_sniffs = extract_agent_face_sniffs(RI2_3_6_boris)
print(agent_sniffs[['Subject', 'Behavior', 'Start (s)', 'Stop (s)']])


          Subject         Behavior  Start (s)  Stop (s)
12   social_agent  facial sniffing     25.600    26.100
23   social_agent  facial sniffing     68.767    69.600
36   social_agent  facial sniffing     91.900    92.267
40   social_agent  facial sniffing    153.900   154.400
41   social_agent  facial sniffing    156.233   156.567
43   social_agent  facial sniffing    195.433   195.767
46   social_agent  facial sniffing    196.167   197.500
53   social_agent  facial sniffing    218.300   219.367
59   social_agent  facial sniffing    226.433   226.900
64   social_agent  facial sniffing    234.867   235.367
67   social_agent  facial sniffing    249.733   250.567
68   social_agent  facial sniffing    257.833   258.367
70   social_agent  facial sniffing    267.400   268.933
71   social_agent  facial sniffing    269.767   270.600
75   social_agent  facial sniffing    273.367   274.200
78   social_agent  facial sniffing    282.033   283.067
79   social_agent  facial sniffing    288.200   

In [ ]:
print(RI2_3_6_boris['Subject'].unique())
print(RI2_3_6_boris['Behavior'].unique())


['subject' 'social_agent']
['facial sniffing' 'body sniffing' 'anogenital sniffing' 'Posturing'
 'fighting' 'chasing' 'tails rattling']
